# Khumbu Icefall Era Appearance (Early vs Late) — NOT a hazard forecast
This notebook shows what the trained CNN sees in a satellite screenshot (pre-2015 vs post-2015 appearance). It does NOT predict avalanches or floods.

In [ ]:
import json
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

### 1. Define the Model Architecture

In [ ]:
class CNN(nn.Module):
    def __init__(self, hidden=128, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(64 * 4 * 4, hidden), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 2),
        )
    def forward(self, x):
        return self.head(self.features(x))

### 2. Load Early Warning Model Weights
We load the normalization stats and the original Early Warning model weights (`outputs/khumbu_cnn.pt`), which learned the structural differences between historically stable ice (pre-2015) and unstable, high-risk ice (post-2015).

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load optimal hyperparameters
results = json.load(open("outputs/nn_results.json"))
best_config = results["best_config"]
hidden_size = best_config["hidden"]
dropout_rate = best_config["dropout"]

# Load normalization stats
stats = json.load(open("outputs/eda_summary.json"))
mean = stats["train_mean"]
std = stats["train_std"]

# Initialize model and load weights
model = CNN(hidden=hidden_size, dropout=dropout_rate).to(device)
model.load_state_dict(torch.load("outputs/khumbu_cnn.pt", map_location=device, weights_only=True))
model.eval()

# Set up the evaluation transform pipeline
IMG_SIZE = 128
eval_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
print("✅ Early Warning Model loaded successfully!")

### 3. Provide an Image to Predict
Specify the path to the satellite image you want to analyze.

In [ ]:
# 👉 INPUT: Set your image path here
IMAGE_PATH = "dataset/no_avalanche/2021.11.13.png"

img = Image.open(IMAGE_PATH).convert("RGB")

plt.figure(figsize=(8, 4))
plt.imshow(img)
plt.axis("off")
plt.title(f"Selected Image: {IMAGE_PATH}")
plt.show()

### 4. Run Early Warning Inference

In [ ]:
# Pre-process the image
input_tensor = eval_tfm(img).unsqueeze(0).to(device)  # Add batch dimension

# Run model
with torch.no_grad():
    outputs = model(input_tensor)
    probabilities = torch.nn.functional.softmax(outputs[0], dim=0)
    prediction_idx = torch.argmax(probabilities).item()

# Index 0 = Early-era appearance (<=2014)
# Index 1 = Late-era appearance (>=2015). This is a visual era proxy,
# NOT a flood/avalanche forecast.
label_names = ["Early-era appearance (<=2014)", "Late-era appearance (>=2015)"]
predicted_label = label_names[prediction_idx]
confidence = probabilities[prediction_idx].item() * 100

print("=" * 70)
print(f"🛰️  ERA APPEARANCE PREDICTION (not a hazard warning):")
print(f"    {predicted_label}")
print("-" * 70)
print(f"The system is {confidence:.2f}% confident in this assessment.")
print("\nAppearance probability:")
print(f"  - {probabilities[0].item()*100:.2f}% Early-era appearance.")
print(f"  - {probabilities[1].item()*100:.2f}% Late-era appearance.")
print("This does NOT predict avalanches or floods.")
print("=" * 70)